
# LangChain 中间件实战教程

> 基于中间件完全指南的代码实践

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rich import print as rprint
import dotenv

dotenv.load_dotenv(override=True)

# 创建 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)
print("LLM 初始化完成")

## 1. Callbacks 回调系统

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult

class LoggingCallback(BaseCallbackHandler):
    """日志回调处理器"""
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        rprint(f"[cyan]LLM 开始[/cyan] 输入: {prompts[0][:50]}...")
    
    def on_llm_new_token(self, token, **kwargs):
        print(token, end="", flush=True)
    
    def on_llm_end(self, response: LLMResult, **kwargs):
        rprint(f"\n[green]LLM 完成[/green]")
    
    def on_llm_error(self, error, **kwargs):
        rprint(f"[red]LLM 错误:[/red] {error}")
    
    def on_chain_start(self, serialized, inputs, **kwargs):
        rprint(f"[yellow]Chain 开始:[/yellow] {serialized.get('name', 'unknown')}")
    
    def on_tool_start(self, serialized, input_str, **kwargs):
        rprint(f"[magenta]工具调用:[/magenta] {serialized.get('name')}")
    
    def on_tool_end(self, output, **kwargs):
        rprint(f"[green]工具结果:[/green] {output[:100]}")

# 使用回调
callback = LoggingCallback()
llm_with_callback = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    callbacks=[callback]
)

rprint("\n[bold]测试回调:[/bold]")
response = llm_with_callback.invoke("你好，请简短回复")

## 2. LCEL 表达式语言 - 管道操作

In [ ]:
# 基础链
prompt = ChatPromptTemplate.from_template("用一句话解释什么是{topic}")
output_parser = StrOutputParser()

# 使用管道操作符组合
chain = prompt | llm | output_parser

rprint("[bold]LCEL 管道链:[/bold]")
result = chain.invoke({"topic": "机器学习"})
rprint(f"结果: {result}")

## 3. RunnablePassthrough - 透传

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 直接透传
passthrough = RunnablePassthrough()
result = passthrough.invoke("hello")
rprint(f"透传结果: {result}")

# 带预处理的透传
def preprocess(input):
    return {"question": input["question"].strip().upper()}

chain = RunnableLambda(preprocess) | prompt | llm | output_parser
result = chain.invoke({"question": " 什么是AI？  "})
rprint(f"预处理后结果: {result}")

## 4. RunnableLambda - 函数转Runnable

In [ ]:
# 将普通函数转为 Runnable
def count_words(text):
    return {"word_count": len(text.split()), "text": text}

counter = RunnableLambda(count_words)
result = counter.invoke("hello world test")
rprint(f"词数统计: {result}")

# 组合使用
def format_output(result):
    return f"回答({result['word_count']}词): {result['text']}"

chain = prompt | llm | output_parser | counter | RunnableLambda(format_output)
result = chain.invoke({"topic": "Python"})
rprint(f"格式化输出: {result}")

## 5. RunnableParallel - 并行执行

In [ ]:
from langchain_core.runnables import RunnableParallel

# 并行执行多个链
summary_prompt = ChatPromptTemplate.from_template("一句话总结：{topic}")
keywords_prompt = ChatPromptTemplate.from_template("列出3个关键词：{topic}")

parallel_chain = RunnableParallel(
    summary=summary_prompt | llm | output_parser,
    keywords=keywords_prompt | llm | output_parser
)

rprint("[bold]并行执行:[/bold]")
result = parallel_chain.invoke({"topic": "人工智能"})
rprint(f"总结: {result['summary']}")
rprint(f"关键词: {result['keywords']}")

## 6. RunnableBranch - 条件分支

In [ ]:
from langchain_core.runnables import RunnableBranch

# 定义不同的处理链
short_prompt = ChatPromptTemplate.from_template("简短回答：{question}")
long_prompt = ChatPromptTemplate.from_template("详细回答：{question}")

# 条件分支
branch = RunnableBranch(
    (lambda x: len(x["question"]) < 10, short_prompt | llm | output_parser),
    (lambda x: len(x["question"]) >= 10, long_prompt | llm | output_parser),
)

rprint("[bold]条件分支:[/bold]")
result = branch.invoke({"question": "你好"})
rprint(f"短问题结果: {result[:100]}...")

result = branch.invoke({"question": "请详细解释一下什么是深度学习以及它的应用场景"})
rprint(f"长问题结果: {result[:100]}...")

## 7. Fallbacks 降级机制

In [ ]:
# 主模型 + 降级模型
primary_llm = ChatOpenAI(
    model="nonexistent-model",  # 故意使用不存在的模型
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

fallback_llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 设置降级
chain_with_fallback = prompt | primary_llm.with_fallbacks([fallback_llm]) | output_parser

rprint("[bold]降级测试:[/bold]")
try:
    result = chain_with_fallback.invoke({"topic": "AI"})
    rprint(f"降级成功: {result[:80]}...")
except Exception as e:
    rprint(f"错误: {e}")

## 8. 重试策略

In [ ]:
# 带重试的链
chain_with_retry = (
    prompt 
    | llm.with_retry(
        stop_after_attempt=3,  # 最多重试3次
        wait_exponential_jitter=True,  # 指数退避
    )
    | output_parser
)

rprint("[bold]重试测试:[/bold]")
result = chain_with_retry.invoke({"topic": "Python"})
rprint(f"结果: {result[:80]}...")

## 9. 缓存机制

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
import time

# 设置内存缓存
set_llm_cache(InMemoryCache())

rprint("[bold]缓存测试:[/bold]")

# 第一次调用
start = time.time()
result1 = llm.invoke("什么是机器学习？")
time1 = time.time() - start
rprint(f"第一次调用: {time1:.2f}秒")

# 第二次调用（使用缓存）
start = time.time()
result2 = llm.invoke("什么是机器学习？")
time2 = time.time() - start
rprint(f"第二次调用: {time2:.2f}秒 (缓存命中)")

rprint(f"加速: {time1/time2:.1f}倍")

## 10. 流式输出

In [ ]:
# 清除缓存以测试流式
set_llm_cache(None)

# 基础流式输出
rprint("[bold]流式输出:[/bold]")
print("回答: ", end="")
for chunk in llm.stream("用一句话介绍Python"):
    print(chunk.content, end="", flush=True)
print()

# 链的流式输出
chain = prompt | llm | StrOutputParser()
rprint("\n[bold]链式流式:[/bold]")
print("回答: ", end="")
for chunk in chain.stream({"topic": "春天"}):
    print(chunk, end="", flush=True)
print()

## 11. RunnableConfig - 运行时配置

In [ ]:
from langchain_core.runnables import RunnableConfig

# 运行时配置
config = RunnableConfig(
    callbacks=[LoggingCallback()],  # 回调
    run_name="test_chain",  # 运行名称
    tags=["test", "demo"],  # 标签
    metadata={"version": "1.0"},  # 元数据
)

rprint("[bold]带配置的调用:[/bold]")
result = chain.invoke({"topic": "数据科学"}, config=config)

## 12. 性能监控回调

In [ ]:
import time

class PerformanceMonitor(BaseCallbackHandler):
    """性能监控回调"""
    
    def __init__(self):
        self.timers = {}
        self.metrics = {}
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.timers['llm'] = time.time()
        rprint("[dim]⏱ LLM 计时开始[/dim]")
    
    def on_llm_end(self, response, **kwargs):
        elapsed = time.time() - self.timers['llm']
        self.metrics['llm_time'] = elapsed
        rprint(f"[dim]⏱ LLM 耗时: {elapsed:.2f}秒[/dim]")
    
    def on_tool_start(self, serialized, input_str, **kwargs):
        self.timers['tool'] = time.time()
    
    def on_tool_end(self, output, **kwargs):
        elapsed = time.time() - self.timers['tool']
        rprint(f"[dim]⏱ 工具耗时: {elapsed:.2f}秒[/dim]")

# 使用性能监控
monitor = PerformanceMonitor()
llm_monitored = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    callbacks=[monitor]
)

rprint("[bold]性能监控:[/bold]")
result = llm_monitored.invoke("你好")
rprint(f"总耗时指标: {monitor.metrics}")

## 13. 完整中间件链示例

In [ ]:
from datetime import datetime

# 1. 定义完整的中间件回调
class ProductionCallback(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        rprint(f"[dim][{datetime.now().strftime('%H:%M:%S')}] 开始处理[/dim]")
    
    def on_llm_end(self, response, **kwargs):
        rprint(f"[dim][{datetime.now().strftime('%H:%M:%S')}] 处理完成[/dim]")
    
    def on_llm_error(self, error, **kwargs):
        rprint(f"[red][{datetime.now().strftime('%H:%M:%S')}] 处理失败: {error}[/red]")

# 2. 预处理和后处理
def preprocess(input):
    return {"question": input["question"].strip()}

def postprocess(output):
    return {"answer": output, "timestamp": datetime.now().isoformat()}

# 3. 创建完整链
full_chain = (
    RunnableLambda(preprocess)
    | prompt
    | llm.with_fallbacks([fallback_llm])
    | output_parser
    | RunnableLambda(postprocess)
)

# 4. 使用配置执行
config = RunnableConfig(
    callbacks=[ProductionCallback()],
    tags=["production"],
    metadata={"user_id": "test_user"},
)

rprint("[bold]完整中间件链:[/bold]")
result = full_chain.invoke({"question": " 什么是深度学习？ "}, config=config)
rprint(f"回答: {result['answer'][:80]}...")
rprint(f"时间戳: {result['timestamp']}")

## 14. 异步回调示例

In [ ]:
import asyncio

class AsyncCallback(BaseCallbackHandler):
    """异步回调处理器"""
    
    async def on_llm_start(self, serialized, prompts, **kwargs):
        rprint("[cyan]异步: LLM 开始[/cyan]")
    
    async def on_llm_end(self, response, **kwargs):
        rprint("[green]异步: LLM 完成[/green]")

# 异步调用
async def async_demo():
    llm_async = ChatOpenAI(
        model=os.getenv("MODEL_NAME"),
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base=os.getenv("OPENAI_BASE_URL"),
        callbacks=[AsyncCallback()]
    )
    
    rprint("[bold]异步调用:[/bold]")
    result = await llm_async.ainvoke("你好")
    rprint(f"结果: {result.content[:50]}...")

# 运行异步示例
await async_demo()

## 总结

### 中间件概念速查

| 概念 | 类/函数 | 用途 |
|------|---------|------|
| 回调 | `BaseCallbackHandler` | 拦截事件 |
| 管道 | `\|` 操作符 | 组合链 |
| 透传 | `RunnablePassthrough` | 直接传递输入 |
| 函数 | `RunnableLambda` | 函数转Runnable |
| 并行 | `RunnableParallel` | 并行执行 |
| 分支 | `RunnableBranch` | 条件执行 |
| 降级 | `with_fallbacks()` | 失败时降级 |
| 重试 | `with_retry()` | 自动重试 |
| 缓存 | `InMemoryCache` | 结果缓存 |
| 流式 | `stream()` | 流式输出 |
| 配置 | `RunnableConfig` | 运行时配置 |
| 监控 | `PerformanceMonitor` | 性能监控 |